# Détection de routes — GPSO

Pipeline complet : **WMS IGN → FLAIR-INC → masque routes → export géospatial**

| Étape | Module |
|-------|--------|
| Téléchargement ortho-photos | `ortho_ign` |
| Segmentation sémantique | `flair_inference` |
| Logique métier routes | ce notebook |

Communes disponibles : `BOULOGNE-BILLANCOURT`, `CHAVILLE`, `ISSY-LES-MOULINEAUX`,
`MARNES-LA-COQUETTE`, `MEUDON`, `SEVRES`, `VANVES`, `VILLE-D'AVRAY`

## 1 · Installation & modèle

In [ ]:
import subprocess, sys, zipfile, urllib.request, pathlib

# Dépendances géospatiales
subprocess.run(
    ["conda", "install", "-c", "conda-forge", "-y", "--quiet",
     "rasterio", "geopandas", "shapely", "fiona", "onnxruntime", "huggingface_hub"],
    check=True,
)

# Téléchargement et extraction du projet (sans git ni pip install)
LIB_DIR = pathlib.Path("/arcgis/home/landcover-lib")  # adapter selon l'environnement
if not LIB_DIR.exists():
    zip_path = pathlib.Path("/arcgis/home/landcover.zip")
    urllib.request.urlretrieve(
        "https://github.com/DIGIT-Seine-Ouest/cv-routes-bitumees/archive/refs/heads/main.zip",
        zip_path,
    )
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(LIB_DIR)

sys.path.insert(0, str(next(LIB_DIR.iterdir())))

## 2 · Configuration

In [ ]:
from flair_inference import download_model, FlairModel

# --- Paramètres à adapter ---
COMMUNE    = "MEUDON"
TILES_DIR  = f"/arcgis/home/tiles/{COMMUNE}"
EXPORT_DIR = f"/arcgis/home/exports/{COMMUNE}"
MODEL_PATH = "/arcgis/home/flair_12cl_resnet34_unet.onnx"

# Classes FLAIR qui définissent une "route"
ROAD_CLASSES = {1, 2}   # pervious_surface + impervious_surface
ROAD_COLOR   = (255, 140, 0)

# URL GeoJSON GPSO (EPSG:2154)
GPSO_GEOJSON_URL = (
    "https://data.seineouest.fr/api/explore/v2.1/catalog/datasets/plu-de-gpso"
    "/exports/geojson?lang=fr&timezone=Europe%2FBerlin&epsg=2154"
)

# Téléchargement du modèle
MODEL_PATH = download_model(dest=MODEL_PATH)
model      = FlairModel(MODEL_PATH)
print(f"Modèle prêt — {model.NUM_CLASSES} classes FLAIR")

## 3 · Téléchargement des tuiles WMS

In [ ]:
from ortho_ign import fetch_city_tiles

def progress(r, m):
    print(f"\r  {'█' * int(r * 30) + '░' * (30 - int(r * 30))}  {m}", end="", flush=True)

tiles = fetch_city_tiles(
    commune_name=COMMUNE,
    tiles_dir=TILES_DIR,
    geojson_url=GPSO_GEOJSON_URL,
    on_progress=progress,
)
print(f"\n{len(tiles)} tuile(s) prête(s)")

## 4 · Inférence FLAIR

In [ ]:
import numpy as np
from flair_inference import mask_from_classes, colorize, apply_overlay, class_stats
from ortho_ign import load_tile_as_rgb
from PIL import Image

results = []

for i, tile_path in enumerate(tiles):
    img    = load_tile_as_rgb(tile_path)
    arr255 = np.transpose(np.array(img, dtype=np.float32), (2, 0, 1))

    class_map  = model.predict(arr255)
    road_mask  = mask_from_classes(class_map, ROAD_CLASSES)
    stats      = class_stats(class_map, target_ids=ROAD_CLASSES)

    results.append({
        "tile_path":    tile_path,
        "ortho":        img,
        "flair_map":    Image.fromarray(colorize(class_map)),
        "road_overlay": apply_overlay(img, road_mask, ROAD_COLOR),
        "road_mask":    road_mask,
        "class_map":    class_map,
        "road_pct":     stats["target_pct"],
        "stats":        stats,
    })

avg_road = sum(r["road_pct"] for r in results) / len(results)
print(f"{len(results)} tuiles — surface route moyenne : {avg_road:.1f}%")

## 5 · Visualisation

In [ ]:
import matplotlib.pyplot as plt

n = min(4, len(results))
fig, axes = plt.subplots(n, 3, figsize=(13, 4 * n))
if n == 1:
    axes = [axes]

for i, r in enumerate(results[:n]):
    axes[i][0].imshow(r["ortho"])
    axes[i][1].imshow(r["flair_map"])
    axes[i][2].imshow(r["road_overlay"])
    axes[i][0].set_ylabel(f"tuile {i}", fontsize=9)
    axes[i][2].set_title(f"{r['road_pct']:.1f}%", fontsize=9)

axes[0][0].set_title("Ortho IGN")
axes[0][1].set_title("FLAIR — 12 classes")
axes[0][2].set_title("Routes détectées")
for row in axes:
    for ax in row:
        ax.axis("off")

plt.suptitle(COMMUNE, fontsize=13)
plt.tight_layout()
plt.show()

## 6 · Export géospatial

In [ ]:
import os
from ortho_ign import mask_to_geotiff, collect_mask_polys, write_polys_to_gpkg, zip_files

tif_dir   = os.path.join(EXPORT_DIR, "tifs")
os.makedirs(tif_dir, exist_ok=True)

tif_paths = []
all_polys = []

for r in results:
    tile_name = os.path.splitext(os.path.basename(r["tile_path"]))[0]
    tif_path  = mask_to_geotiff(
        r["road_mask"], r["tile_path"],
        os.path.join(tif_dir, f"{tile_name}_routes.tif"),
    )
    tif_paths.append(tif_path)
    all_polys = collect_mask_polys(r["road_mask"], r["tile_path"], all_polys)

gpkg_path = write_polys_to_gpkg(
    {"routes": all_polys},
    os.path.join(EXPORT_DIR, f"{COMMUNE.lower()}_routes.gpkg"),
)
zip_path = zip_files(
    tif_paths,
    os.path.join(EXPORT_DIR, f"{COMMUNE.lower()}_routes_tifs.zip"),
    tif_dir,
)

print(f"GeoPackage : {gpkg_path}")
print(f"ZIP TIFFs  : {zip_path}")

## 7 · Résultats vectoriels

In [ ]:
import geopandas as gpd

gdf = gpd.read_file(gpkg_path, layer="routes")
print(f"Polygones : {len(gdf)}  |  CRS : {gdf.crs}  |  Surface : {gdf.area.sum() / 1e4:.1f} ha")

fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, color="orange", alpha=0.75, edgecolor="none")
ax.set_title(f"Routes détectées — {COMMUNE}")
plt.tight_layout()
plt.show()